
# Unix Shell

In this project, you'll build a simple Unix shell. The shell is the heart of
the command-line interface, and thus is central to the Unix/C programming
environment. Mastering use of the shell is necessary to become proficient in
this world; knowing how the shell itself is built is the focus of this
project.

There are three specific objectives to this assignment:

* To further familiarize yourself with the Linux programming environment.
* To learn how processes are created, destroyed, and managed.
* To gain exposure to the necessary functionality in shells.

## Overview

In this assignment, you will implement a *command line interpreter (CLI)* or,
as it is more commonly known, a *shell*. The shell should operate in this
basic way: when you type in a command (in response to its prompt), the shell
creates a child process that executes the command you entered and then prompts
for more user input when it has finished.

The shells you implement will be similar to, but simpler than, the one you run
every day in Unix. If you don't know what shell you are running, it's probably
`bash`. One thing you should do on your own time is learn more about your
shell, by reading the man pages or other online materials.

## Program Specifications

### Basic Shell: `wish`

Your basic shell, called `wish` (short for Wisconsin Shell, naturally), is
basically an interactive loop: it repeatedly prints a prompt `wish> ` (note
the space after the greater-than sign), parses the input, executes the command
specified on that line of input, and waits for the command to finish. This is
repeated until the user types `exit`.  The name of your final executable
should be `wish`.

The shell can be invoked with either no arguments or a single argument;
anything else is an error. Here is the no-argument way:

```
prompt> ./wish
wish> 
```

At this point, `wish` is running, and ready to accept commands. Type away!

The mode above is called *interactive* mode, and allows the user to type
commands directly. The shell also supports a *batch mode*, which instead reads
input from a batch file and executes commands from therein. Here is how you
run the shell with a batch file named `batch.txt`:

```
prompt> ./wish batch.txt
```

One difference between batch and interactive modes: in interactive mode, a
prompt is printed (`wish> `). In batch mode, no prompt should be printed.

You should structure your shell such that it creates a process for each new
command (the exception are *built-in commands*, discussed below).  Your basic
shell should be able to parse a command and run the program corresponding to
the command.  For example, if the user types `ls -la /tmp`, your shell should
run the program `/bin/ls` with the given arguments `-la` and `/tmp` (how does
the shell know to run `/bin/ls`? It's something called the shell **path**;
more on this below).

## Structure

### Basic Shell

The shell is very simple (conceptually): it runs in a while loop, repeatedly
asking for input to tell it what command to execute. It then executes that
command. The loop continues indefinitely, until the user types the built-in
command `exit`, at which point it exits. That's it!

For reading lines of input, you should use `getline()`. This allows you to
obtain arbitrarily long input lines with ease. Generally, the shell will be
run in *interactive mode*, where the user types a command (one at a time) and
the shell acts on it. However, your shell will also support *batch mode*, in
which the shell is given an input file of commands; in this case, the shell
should not read user input (from `stdin`) but rather from this file to get the
commands to execute.

In either mode, if you hit the end-of-file marker (EOF), you should call
`exit(0)` and exit gracefully. 

To parse the input line into constituent pieces, you might want to use
`strsep()`. Read the man page (carefully) for more details.

To execute commands, look into `fork()`, `exec()`, and `wait()/waitpid()`.
See the man pages for these functions, and also read the relevant [book
chapter](http://www.ostep.org/cpu-api.pdf) for a brief overview.

You will note that there are a variety of commands in the `exec` family; for
this project, you must use `execv`. You should **not** use the `system()`
library function call to run a command.  Remember that if `execv()` is
successful, it will not return; if it does return, there was an error (e.g.,
the command does not exist). The most challenging part is getting the
arguments correctly specified. 

### Paths

In our example above, the user typed `ls` but the shell knew to execute the
program `/bin/ls`. How does your shell know this?

It turns out that the user must specify a **path** variable to describe the
set of directories to search for executables; the set of directories that
comprise the path are sometimes called the *search path* of the shell. The
path variable contains the list of all directories to search, in order, when
the user types a command. 

**Important:** Note that the shell itself does not *implement* `ls` or other
commands (except built-ins). All it does is find those executables in one of
the directories specified by `path` and create a new process to run them.

To check if a particular file exists in a directory and is executable,
consider the `access()` system call. For example, when the user types `ls`,
and path is set to include both `/bin` and `/usr/bin`, try `access("/bin/ls",
X_OK)`. If that fails, try "/usr/bin/ls". If that fails too, it is an error.

Your initial shell path should contain one directory: `/bin`

Note: Most shells allow you to specify a binary specifically without using a
search path, using either **absolute paths** or **relative paths**. For
example, a user could type the **absolute path** `/bin/ls` and execute the
`ls` binary without a search path being needed. A user could also specify a
**relative path** which starts with the current working directory and
specifies the executable directly, e.g., `./main`. In this project, you **do
not** have to worry about these features.

### Built-in Commands

Whenever your shell accepts a command, it should check whether the command is
a **built-in command** or not. If it is, it should not be executed like other
programs. Instead, your shell will invoke your implementation of the built-in
command. For example, to implement the `exit` built-in command, you simply
call `exit(0);` in your wish source code, which then will exit the shell.

In this project, you should implement `exit`, `cd`, and `path` as built-in
commands.

* `exit`: When the user types `exit`, your shell should simply call the `exit`
  system call with 0 as a parameter. It is an error to pass any arguments to
  `exit`. 

* `cd`: `cd` always take one argument (0 or >1 args should be signaled as an
error). To change directories, use the `chdir()` system call with the argument
supplied by the user; if `chdir` fails, that is also an error.

* `path`: The `path` command takes 0 or more arguments, with each argument
  separated by whitespace from the others. A typical usage would be like this:
  `wish> path /bin /usr/bin`, which would add `/bin` and `/usr/bin` to the
  search path of the shell. If the user sets path to be empty, then the shell
  should not be able to run any programs (except built-in commands). The
  `path` command always overwrites the old path with the newly specified
  path. 

### Redirection

Many times, a shell user prefers to send the output of a program to a file
rather than to the screen. Usually, a shell provides this nice feature with
the `>` character. Formally this is named as redirection of standard
output. To make your shell users happy, your shell should also include this
feature, but with a slight twist (explained below).

For example, if a user types `ls -la /tmp > output`, nothing should be printed
on the screen. Instead, the standard output of the `ls` program should be
rerouted to the file `output`. In addition, the standard error output of
the program should be rerouted to the file `output` (the twist is that this
is a little different than standard redirection).

If the `output` file exists before you run your program, you should simple
overwrite it (after truncating it).  

The exact format of redirection is a command (and possibly some arguments)
followed by the redirection symbol followed by a filename. Multiple
redirection operators or multiple files to the right of the redirection sign
are errors.

Note: don't worry about redirection for built-in commands (e.g., we will
not test what happens when you type `path /bin > file`).

### Parallel Commands

Your shell will also allow the user to launch parallel commands. This is
accomplished with the ampersand operator as follows:

```
wish> cmd1 & cmd2 args1 args2 & cmd3 args1
```

In this case, instead of running `cmd1` and then waiting for it to finish,
your shell should run `cmd1`, `cmd2`, and `cmd3` (each with whatever arguments
the user has passed to it) in parallel, *before* waiting for any of them to
complete. 

Then, after starting all such processes, you must make sure to use `wait()`
(or `waitpid`) to wait for them to complete. After all processes are done,
return control to the user as usual (or, if in batch mode, move on to the next
line).


### Program Errors

**The one and only error message.** You should print this one and only error
message whenever you encounter an error of any type:

```
    char error_message[30] = "An error has occurred\n";
    write(STDERR_FILENO, error_message, strlen(error_message)); 
```

The error message should be printed to stderr (standard error), as shown
above. 

After most errors, your shell simply *continue processing* after
printing the one and only error message. However, if the shell is
invoked with more than one file, or if the shell is passed a bad batch
file, it should exit by calling `exit(1)`.

There is a difference between errors that your shell catches and those that
the program catches. Your shell should catch all the syntax errors specified
in this project page. If the syntax of the command looks perfect, you simply
run the specified program. If there are any program-related errors (e.g.,
invalid arguments to `ls` when you run it, for example), the shell does not
have to worry about that (rather, the program will print its own error
messages and exit).


### Miscellaneous Hints

Remember to get the **basic functionality** of your shell working before
worrying about all of the error conditions and end cases. For example, first
get a single command running (probably first a command with no arguments, such
as `ls`). 

Next, add built-in commands. Then, try working on redirection. Finally, think
about parallel commands. Each of these requires a little more effort on
parsing, but each should not be too hard to implement.

At some point, you should make sure your code is robust to white space of
various kinds, including spaces (` `) and tabs (`\t`). In general, the user
should be able to put variable amounts of white space before and after
commands, arguments, and various operators; however, the operators
(redirection and parallel commands) do not require whitespace.

Check the return codes of all system calls from the very beginning of your
work. This will often catch errors in how you are invoking these new system
calls. It's also just good programming sense.

Beat up your own code! You are the best (and in this case, the only) tester of
this code. Throw lots of different inputs at it and make sure the shell
behaves well. Good code comes through testing; you must run many different
tests to make sure things work as desired. Don't be gentle -- other users
certainly won't be. 

Finally, keep versions of your code. More advanced programmers will use a
source control system such as git. Minimally, when you get a piece of
functionality working, make a copy of your .c file (perhaps a subdirectory
with a version number, such as v1, v2, etc.). By keeping older, working
versions around, you can comfortably work on adding new functionality, safe in
the knowledge you can always go back to an older, working version if need be.


In [2]:
!ls -lsah

total 176K
   0 drwxr-sr-x 3 solveit solveit   88 Nov 14 21:09 .
   0 drwxr-sr-x 3 solveit solveit   27 Nov 12 17:26 ..
   0 drwxr-sr-x 8 solveit solveit  193 Nov 14 21:09 .git
156K -rw-r--r-- 1 solveit solveit 153K Nov 14 21:09 build_wish.ipynb
 16K -rwxr-xr-x 1 solveit solveit  16K Nov 12 20:10 wish
4.0K -rw-r--r-- 1 solveit solveit  482 Nov 12 20:09 wish.c


In [ ]:
%%prompt
How to add this https://github.com/jjmaturino/playful-cli.git as remote origin and sync.

Your folder already has a Git repository (`.git`). Add the GitHub remote and fetch its history:

```bash
git remote add origin https://github.com/jjmaturino/playful-cli.git
git fetch origin
git status
```

If your local changes aren’t committed yet:

```bash
git add wish.c build_wish.ipynb
git commit -m "Add wish shell project"
```

Then push your current branch and set its upstream:

```bash
git push -u origin HEAD
```

After that, the usual sync cycle is:

```bash
git pull --rebase
git push
```

**If `origin` already exists**, replace its URL with `git remote set-url origin https://github.com/jjmaturino/playful-cli.git`.

**If the initial push is rejected**, stop rather than force-pushing: GitHub may already contain commits that need to be reconciled with your local history.

```json {.usage}
{"model":"codex/gpt-6.1-sol","prompt_tokens":34033,"completion_tokens":328,"total_tokens":34361,"cached_tokens":3968,"reasoning_tokens":129,"cost":0.00535095}
```


In [3]:
!git fetch origin


In [4]:
!git status


On branch main

No commits yet

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	build_wish.ipynb
	wish
	wish.c

nothing added to commit but untracked files present (use "git add" to track)


## Problem 1 — A C program that prints its arguments

Create `wish.c` with a `main` function that receives `argc` and `argv`. Compile it into an executable named `wish`.

**Your task:** Print the argument count, then print each argument with its index, including `argv[0]`. This is only an argument-inspection program for now, not a shell.

**Completion checks:**
- Run `./wish` and identify what `argv[0]` contains.
- Run `./wish hello world` and check the count and order.
- Run `./wish "hello world"` and explain why it receives fewer arguments than the previous invocation.
- Compile with warnings enabled and resolve any warnings.

Keep building on this file in the following problems. Save a working version after each milestone.

In [43]:
%%writefile wish.c
#include <stdio.h>

int main(int argc, char *argv[]) {
    printf("argument count: %d\n", argc);

    for (int i = 0; i < argc; i++){
        printf("%d: %s\n", i, argv[i]);
    }


    return 0;
}

Overwriting wish.c


In [44]:
!gcc -Wall -Werror wish.c -o wish


In [45]:
!./wish hello world

argument count: 3
0: ./wish
1: hello
2: world


In [46]:
!./wish "hello world"

argument count: 2
0: ./wish
1: hello world


## Problem 2 — Read lines in an interactive loop

Replace the startup argument-printing behavior with a loop that prints the exact prompt `wish> ` and reads one line from standard input using `getline()`.

**Your task:** Temporarily echo each line back so you can verify input handling. Keep reading until EOF; then exit successfully. Make sure the prompt is visible before waiting for input.

**Completion checks:**
- Enter several lines and get a fresh prompt after each.
- Empty lines do not end the program.
- A very long line is read without being cut off by a fixed-size buffer.
- EOF exits with status 0.

No command execution yet. The echo is temporary debugging output.

In [ ]:
%%prompt
How to do a while loop.

In C, a `while` loop repeats its body **while its condition is true**:

```c
while (condition) {
    // work to repeat
}
```

The condition is checked **before each iteration**, so if it starts false, the body never runs.

For your shell, you can use `while (1)` to keep looping, then use `break;` inside the loop when you reach EOF.

Your loop’s steps will be:
1. Print `wish> `.
2. Flush standard output with `fflush(stdout);` so the prompt is visible.
3. Read a line.
4. If you reached EOF, break out of the loop; otherwise, echo the line.

```json {.usage}
{"model":"codex/gpt-6-astra","prompt_tokens":34516,"completion_tokens":148,"total_tokens":34664,"cached_tokens":33792,"cost":0.0052884}
```


In [50]:
%%writefile wish.c
#include <stdio.h>

int main(int argc, char *argv[]) {
    printf("argument count: %d\n", argc);

    for (int i = 0; i < argc; i++){
        printf("%d: %s\n", i, argv[i]);
    }
    while (1) {
        printf("wish> ");
        fflush(stdout);

    }


    return 0;
}

Overwriting wish.c


In [ ]:
%%prompt
What is fflush and what does it doe?

`printf` can place its output in a **buffer**—temporary memory—rather than immediately displaying it.

`fflush(stdout)` tells C: **send any buffered standard output now**.

Your prompt has no newline (`\n`), so flush it explicitly before waiting for input:

```c
printf("wish> ");
fflush(stdout);
```

This makes sure the user sees the prompt before being asked to type.

Your current loop doesn’t read input yet, so it will print prompts continuously rather than wait. Also, use `while (1)` for now; `true` requires `<stdbool.h>` in C versions before C23.

```json {.usage}
{"model":"codex/gpt-6-astra","prompt_tokens":34833,"completion_tokens":135,"total_tokens":34968,"cached_tokens":34432,"cost":0.005326200000000001}
```


In [ ]:
%%prompt
Does printf flush implicitly if a newline character is placed?

**Usually when printing to a terminal, but not always.** It depends on how `stdout` is buffered:

- **Line-buffered** (typically a terminal): a newline causes buffered output to be sent.
- **Fully buffered** (typically output redirected to a file or pipe): a newline does **not** force a flush.

So it’s not `printf` itself that guarantees flushing on `\n`—it’s the buffering mode of `stdout`.

Since your `wish> ` prompt has no newline, explicitly calling `fflush(stdout)` ensures it appears before you wait for input.

```json {.usage}
{"model":"codex/gpt-6.1-sol","prompt_tokens":35036,"completion_tokens":125,"total_tokens":35161,"cached_tokens":3968,"cost":0.00534915}
```


In [ ]:
%%prompt
Explain line buffered vs fully buffered.

Both modes temporarily collect output in memory, but differ in **when they send it onward**:

| Mode | When output is sent | Typical use |
|---|---|---|
| **Line-buffered** | When a newline is written or the buffer fills | `stdout` connected to a terminal |
| **Fully buffered** | When the buffer fills; a newline has no special effect | `stdout` redirected to a file or pipe |

In **either mode**, `fflush(stdout)` explicitly sends buffered output, and normal program termination flushes it too.

For example, imagine these two calls happen several seconds apart:

```c
printf("Hello");
printf(" world\n");
```

With line buffering, you typically see nothing until the second call, then see `Hello world` together. With full buffering, even the second call might leave everything in memory.

Buffering reduces the overhead of sending output in many tiny pieces. Line buffering balances that efficiency with showing terminal users complete lines promptly.

```json {.usage}
{"model":"codex/gpt-6.1-sol","prompt_tokens":35238,"completion_tokens":205,"total_tokens":35443,"cached_tokens":8320,"cost":0.005439450000000001}
```


In [ ]:
%%prompt
how to define each version

Use `setvbuf()` from `<stdio.h>` to choose the buffering mode for a stream.

**Line-buffered:**
```c
setvbuf(stdout, NULL, _IOLBF, BUFSIZ);
```

**Fully buffered:**
```c
setvbuf(stdout, NULL, _IOFBF, BUFSIZ);
```

Here:
- `stdout` is the stream you’re configuring.
- `NULL` lets the C library manage the buffer.
- `_IOLBF` means line buffering; `_IOFBF` means full buffering.
- `BUFSIZ` is a standard buffer-size constant.

Call it **before any output to `stdout`**, including your first `printf`. It returns `0` on success and nonzero on failure.

For your shell, you don’t need to change the buffering mode—printing the prompt and calling `fflush(stdout)` is sufficient.

```json {.usage}
{"model":"codex/gpt-6.1-sol","prompt_tokens":35501,"completion_tokens":191,"total_tokens":35692,"cached_tokens":35200,"cost":0.0054684}
```


In [ ]:
%%prompt
What do I need to do next?

Next, **read a line inside your loop** so it waits for input instead of printing prompts continuously.

Before the loop, initialize a buffer pointer and its capacity:

```c
char *line = NULL;
size_t capacity = 0;
```

Then, immediately after `fflush(stdout);`, add:

```c
if (getline(&line, &capacity, stdin) == -1) break;
printf("%s", line);
```

`getline` allocates and grows the buffer as needed. It returns `-1` when no line can be read, including at EOF. A blank line still counts as a line, so it won’t stop the loop.

Also remove the old argument-printing code. After the loop, call `free(line);` to release the buffer, and add `#include <stdlib.h>` for `free`.

```json {.usage}
{"model":"codex/gpt-6.1-sol","prompt_tokens":35800,"completion_tokens":178,"total_tokens":35978,"cached_tokens":8320,"cost":0.005503500000000001}
```


In [ ]:
%%prompt
Discuss what getline does... so the buffer is none, and capacity being 0  (both grow)?

Yes—`getline` manages the buffer for you, but **`line` and `capacity` describe different things**:

```c
char *line = NULL;
size_t capacity = 0;
```

- **`line = NULL`** means the pointer does not yet point to a buffer. (`NULL` is C’s equivalent here of “no pointer,” rather than Python’s `None`.)
- **`capacity = 0`** means you haven’t provided any buffer space.

When you call `getline(&line, &capacity, stdin)`, it:
1. Allocates a buffer and updates `line` to point to it.
2. Updates `capacity` to the number of bytes allocated.
3. Stores the input there, including the newline if one was read, followed by a terminating `'\0'`.

On later calls, it **reuses that buffer**. If a line needs more space, it grows the allocation, updates `capacity`, and may change where `line` points.

So **the buffer’s capacity grows as needed; the pointer itself doesn’t “grow”—it can change address**.

The return value is separate: it tells you how many characters were read, including the newline but excluding `'\0'`. That is usually less than the allocated capacity.

```json {.usage}
{"model":"codex/gpt-6.1-sol","prompt_tokens":36010,"completion_tokens":269,"total_tokens":36279,"cached_tokens":35584,"cost":0.005603250000000001}
```


In [ ]:
%%prompt
What is capacity variable typically used for?

`capacity` tracks **how many bytes the buffer can hold**, not how many characters are currently in it.

For example:
- Allocated buffer: **128 bytes** → `capacity` is `128`.
- Input: `"hi\n"` → `getline` returns `3`.
- Stored string: `'h'`, `'i'`, `'\n'`, `'\0'` → occupies **4 bytes**.

On the next call, `getline` uses `capacity` to decide whether the existing buffer has enough room or needs to grow.

That’s why you keep `line` and `capacity` between loop iterations: together, they let `getline` reuse the allocated memory.

```json {.usage}
{"model":"codex/gpt-6.1-sol","prompt_tokens":36340,"completion_tokens":144,"total_tokens":36484,"cached_tokens":35840,"cost":0.005559000000000001}
```


In [ ]:
%%prompt
How would I stop the loop?

Use `break;` to leave the loop when `getline` returns `-1`:

```c
if (getline(&line, &capacity, stdin) == -1) break;
```

For this stage, **EOF stops the loop**:
- In a terminal, press **Ctrl+D on an empty input line**.
- When reading a file, EOF occurs when there’s no more input.

A blank line won’t stop it: pressing Enter produces `"\n"`, which is valid input.

After `break`, execution continues after the loop, where you should `free(line);` and then `return 0;`.

```json {.usage}
{"model":"codex/gpt-6.1-sol","prompt_tokens":36543,"completion_tokens":134,"total_tokens":36677,"cached_tokens":36352,"cost":0.0055819500000000005}
```


In [76]:
%%writefile wish.c
#include <stdio.h>
#include <stdlib.h>

int main(int argc, char *argv[]) {

    char *line = NULL;
    size_t capacity = 0;

    while (1) {
        printf("wish> ");
        fflush(stdout);
        if (getline(&line, &capacity, stdin) == -1) break;
        printf("%s \n", line);
    }


    return 0;
}

Overwriting wish.c


In [77]:
!gcc -Wall -Werror wish.c -o wish

In [ ]:
%%prompt
How can I run this interactively?

`!wish` ran a different installed program named `wish`—not the executable you compiled. **`./wish`** means “run `wish` from the current directory.”

For a genuinely interactive session, open a terminal, change to the directory containing `wish.c`, and run:

```bash
./wish
```

You should see `wish> ` and be able to type lines. Press **Ctrl+D on an empty line** to signal EOF and exit.

Running `!./wish` in a notebook-style code message may not support interactive keyboard input. You can still test its input loop there by piping in lines:

```bash
printf 'hello\nworld\n' | ./wish
```

That supplies two lines, then EOF.

```json {.usage}
{"model":"codex/gpt-6.1-sol","prompt_tokens":37007,"completion_tokens":196,"total_tokens":37203,"cached_tokens":36480,"reasoning_tokens":38,"cost":0.0056980500000000005}
```


In [78]:
!printf 'hello\nworld\n' | ./wish

wish> hello
 
wish> world
 
wish> 

In [79]:
!printf '' | ./wish

wish> 

In [ ]:
%%prompt
Add a test for a really long string and an eof exits

Recompile first—the argument-count output suggests your executable is older than your current source:

```bash
!gcc -Wall -Werror wish.c -o wish
```

Then run these tests in a Python code message. `subprocess.run` sends the input and closes the input pipe, giving your program EOF:

```python
import subprocess

long_line = "x" * 100_000
result = subprocess.run(
    ["./wish"], input=long_line + "\n",
    capture_output=True, text=True, timeout=5
)
assert long_line in result.stdout, "Long line was cut off"
assert result.returncode == 0, "Did not exit successfully after EOF"

result = subprocess.run(
    ["./wish"], input="", capture_output=True, text=True, timeout=5
)
assert result.returncode == 0, "Did not exit successfully on immediate EOF"
print("Long-line and EOF tests passed")
```

The timeout also catches a program that gets stuck rather than exiting.

Two small fixes remain in your C code: use `printf("%s", line);` to avoid adding extra whitespace, and put `free(line);` after the loop.

```json {.usage}
{"model":"codex/gpt-6.1-sol","prompt_tokens":37262,"completion_tokens":288,"total_tokens":37550,"cached_tokens":36480,"reasoning_tokens":39,"cost":0.0058053}
```


In [ ]:
%%prompt
Discuss the subprocess api and what I should understand about it :)

Python’s `subprocess` module lets you **start another program, supply its input, and inspect its output and exit status**. Here it acts as a test driver for your C shell.

The key call in your test is:

```python
result = subprocess.run(
    ["./wish"], input="hello\n", capture_output=True, text=True, timeout=5
)
```

Here’s what each part means:

- **`["./wish"]`**: the program and its arguments, as a list. For example, `["./wish", "batch.txt"]` passes one argument to `wish`. Python doesn’t interpret shell operators such as `|` or `>` here.
- **`input="hello\n"`**: sends these characters to the program’s standard input, then closes that input pipe. Your program reads `hello`, then encounters **EOF** on its next read.
- **`capture_output=True`**: collects both standard output and standard error instead of displaying them.
- **`text=True`**: uses Python strings for input and output rather than bytes.
- **`timeout=5`**: raises a `TimeoutExpired` exception if the program takes longer than five seconds.

`run()` **waits for the program to finish**, then returns a `CompletedProcess` object:

| Attribute | What it tells you |
|---|---|
| `result.stdout` | What the program wrote to standard output |
| `result.stderr` | What it wrote to standard error |
| `result.returncode` | Its exit status: `0` conventionally means success |

One important distinction: **a nonzero exit status does not raise an exception by default**. That lets your tests inspect it with an assertion. Adding `check=True` makes `run()` raise an exception on a nonzero status instead.

For your EOF test, `input=""` means **send no characters and immediately close the pipe**—not “leave the program waiting for keyboard input.”

```json {.usage}
{"model":"codex/gpt-6.1-sol","prompt_tokens":37574,"completion_tokens":412,"total_tokens":37986,"cached_tokens":37120,"cost":0.005945100000000001}
```


In [80]:
import subprocess

In [81]:
r = subprocess.run(
    ["./wish"],
    input="hello\n",
    capture_output=True,
    text=True,
    timeout=5)
r

CompletedProcess(args=['./wish'], returncode=0, stdout='wish> hello\n \nwish> ', stderr='')

In [82]:
r = subprocess.run(
    ["./wish"],
    input="hello\nhow\nare\nyou?",
    capture_output=True,
    text=True,
    timeout=5)
r

CompletedProcess(args=['./wish'], returncode=0, stdout='wish> hello\n \nwish> how\n \nwish> are\n \nwish> you? \nwish> ', stderr='')

In [83]:
r.stdout

'wish> hello\n \nwish> how\n \nwish> are\n \nwish> you? \nwish> '

In [ ]:
%%prompt
Is there a way to display stdout as if the terminal would?

Yes—use `print()` to display the captured string with its newlines:

```python
print(r.stdout, end="")
```

`end=""` avoids adding an extra newline that wasn’t in the output.

To display output **as the program runs**, rather than capture it, omit `capture_output=True`. It will inherit the parent process’s output streams; this does not give it a full interactive terminal.

```json {.usage}
{"model":"codex/gpt-6.1-sol","prompt_tokens":38380,"completion_tokens":87,"total_tokens":38467,"cached_tokens":37760,"cost":0.005822250000000001}
```


In [84]:
print(r.stdout, end="")

wish> hello
 
wish> how
 
wish> are
 
wish> you? 
wish> 

In [ ]:
%%prompt
is there a way to have like r.display_output? 

`CompletedProcess` doesn’t have a `display_output()` method. It stores the results; you choose how to display them.

For captured standard output, the usual approach is:

```python
print(r.stdout, end="")
```

`r.stdout` is already the output string—there’s no additional decoding needed because you used `text=True`.

```json {.usage}
{"model":"codex/gpt-6.1-sol","prompt_tokens":38538,"completion_tokens":74,"total_tokens":38612,"cached_tokens":36480,"cost":0.005836200000000001}
```


In [ ]:
%%prompt
But extending it?

Yes—you can subclass `CompletedProcess` and add a method:

```python
class DisplayProcess(subprocess.CompletedProcess):
    def display_output(self):
        print(self.stdout, end="")
```

However, `subprocess.run()` still returns the original `CompletedProcess` type, not your subclass. You’d need to wrap its result in your class.

For just this convenience, a small helper function is simpler than subclassing. Subclassing becomes more worthwhile if you want several related features, such as displaying stderr or summarizing test failures.

```json {.usage}
{"model":"codex/gpt-6.1-sol","prompt_tokens":38670,"completion_tokens":113,"total_tokens":38783,"cached_tokens":38400,"cost":0.005885250000000002}
```


In [85]:
def display_output(res: subprocess.CompletedProcess) -> None: print(res.stdout, end="")

In [86]:
display_output(r)

wish> hello
 
wish> how
 
wish> are
 
wish> you? 
wish> 

In [90]:
long_line = "x" * 100_000
r = subprocess.run(
    ["./wish"],
    input=long_line + "\n",
    capture_output=True,
    text=True,
    timeout=5,
    check=True
    )

In [92]:
display_output(r)

wish> ...
 
wish> 

In [93]:
r = subprocess.run(
    ["./wish"],
    input="hello\nhow\nare\nyou?",
    text=True,
    timeout=5)
r

CompletedProcess(args=['./wish'], returncode=0)

## Problem 3 — Select interactive or batch input

Use the startup arguments to choose an input source:
- No extra arguments: read standard input and print prompts.
- One extra argument: open that batch file and read it without printing prompts.
- More than one extra argument, or a batch file that cannot be opened: report an error and exit with status 1.

Use the same reading loop for both modes. EOF in either mode exits with status 0.

**Error rule from now on:** Every shell-detected error must write exactly `An error has occurred\n` to standard error, with `\n` meaning a newline. Do not add explanations or use a different error message. Check return values as you introduce operations that can fail.

**Completion checks:** Try a multi-line batch file, an empty file, a nonexistent file, and two filename arguments. Keep the temporary line echo only for debugging.

In [95]:
%%writefile wish.c
#include <stdio.h>
#include <stdlib.h>

int interactive_mode(){
    char *line = NULL;
    size_t capacity = 0;
    while (1){
        printf("wish> ");
        fflush(stdout);
        if (getline(&line, &capacity, stdin) == -1) break;
        printf("%s \n", line);
    }


    return 0;
}

int batch_mode(){

    printf("batch mode!!!!\n");
    return 0;
}


int main(int argc, char *argv[]) {

    if (argc == 1){ interactive_mode();}
    else { batch_mode();}


    return 0;
}

Overwriting wish.c


In [96]:
!gcc -Wall -Werror wish.c -o wish

In [97]:
!./wish hello

batch mode!!!!


## Problem 4 — Turn a command line into arguments

Parse each input line into a command name and its arguments. Consider `strsep()` and how you will handle empty tokens.

**Your task:** Separate tokens on spaces, tabs, and the line-ending newline. Ignore whitespace-only input. Prepare an argument array with a terminating null pointer for later use with `execv()`.

Temporarily print the parsed tokens instead of echoing the entire line.

**Completion checks:**
- `ls -la /tmp` produces three tokens in order.
- Leading/trailing whitespace, repeated spaces, and tabs do not create unwanted empty arguments.
- A blank line causes no error.
- You have a clear plan for the lifetime of the input buffer and pointers into it.

Do not implement quoting, wildcard expansion, or variable expansion; the assignment does not require them. The quoted startup argument in Problem 1 was processed by the shell that launched your program.

## Problem 5 — Execute one external command and wait

Replace token-debugging output with execution. For this milestone only, look for commands under `/bin`.

**Your task:** Construct the executable pathname, check it with `access(..., X_OK)`, create a child with `fork()`, and execute the command in that child using `execv()`. The parent waits with `wait()` or `waitpid()` before reading another line.

Use `execv()`, not `system()` or a different exec-family function. If execution fails in the child, report the error and terminate that child; it must not become another shell loop.

**Completion checks:**
- `ls` and `ls -la /tmp` run correctly.
- `sleep 1` finishes before the next prompt appears.
- An unknown command reports the required error, and the shell accepts another command.
- A failed command does not terminate the parent shell.
- No temporary argument/token/line debugging output remains.

## Problem 6 — Implement `exit` and `cd`

Recognize built-ins before searching for an external executable. Run built-ins in the shell process itself.

**Your task:**
- `exit` with no arguments exits with status 0.
- `exit` with any arguments reports an error and leaves the shell running.
- `cd` requires exactly one argument and calls `chdir()`.
- Wrong argument counts or a failed `chdir()` report an error and leave the shell running.

**Completion checks:**
- Run `pwd`, then `cd /tmp`, then `pwd` again.
- Try `cd`, `cd /tmp /bin`, and `cd` with a nonexistent directory.
- Try `exit extra`, followed by a valid command, then `exit`.
- In a batch file, valid `exit` prevents subsequent lines from running.

Explain why running `cd` in a child would not accomplish the required behavior.

## Problem 7 — Maintain the shell's search path

Replace the hard-coded `/bin` lookup with a shell-owned list of directories. Initialize it to exactly one entry: `/bin`.

**Your task:** Implement the `path` built-in. It accepts zero or more directory arguments and replaces the entire previous list. Search directories in order for external commands using `access(..., X_OK)`.

An empty path disables external commands, but not built-ins. Keep path entries valid after the input buffer is reused or freed; release old entries when replacing the path.

**Completion checks:**
- `path /bin /usr/bin` allows commands to be found in either directory.
- `path` followed by `ls` reports an error; `cd`, `path`, and `exit` still work.
- `path /bin` restores external execution.
- Replacing the path does not retain old directories.
- `path /nonexistent /bin` still lets you find `ls` in the later directory.

Direct execution via absolute or relative command paths is not required.

## Problem 8 — Parse and validate output redirection

Extend parsing to recognize `>` even when there is no whitespace around it. Separate the command's argument list from an optional output filename.

**Your task:** Before launching a command, reject missing commands, missing output filenames, multiple `>` operators, and multiple tokens to the right of `>`.

**Completion checks:**
- Accept `ls > output`, `ls>output`, and `ls -la /tmp>output` as equivalent redirection forms.
- Reject `> output`, `ls >`, `ls > one two`, and `ls > one > two`.
- Blank input remains harmless.
- Invalid syntax reports the required error without launching the malformed command.

At this checkpoint, you are validating the parsed representation; actual file redirection comes next. Redirection for built-ins is outside the required tests.

## Problem 9 — Redirect both output streams

Use the parsed filename to redirect an external command's standard output **and standard error** to the same file.

**Your task:** In the child, open the destination for writing, creating it if needed and truncating it if it already exists. Connect the file to both output descriptors before `execv()`. Study `open()`, `dup2()`, and `close()` for this step, and handle failures.

Keep the parent shell's descriptors unchanged.

**Completion checks:**
- `echo hello>output` writes to the file, not the terminal.
- Repeating redirection with shorter output removes old trailing content.
- `ls /a-directory-that-does-not-exist > errors` puts the program's diagnostic in the file.
- A following command without redirection still prints normally.
- An output pathname in a nonexistent directory produces the required shell error, and the shell continues.

Do not replace diagnostics generated by an executed program with the shell's generic error message.

## Problem 10 — Parse a line into multiple commands

Add support for `&` as a separator between commands. It must work with or without surrounding whitespace.

**Your task:** Split the line into command segments, then apply your existing argument and redirection parsing to each segment independently. Keep parsing separate from launching so the next problem can change when waiting happens.

**Completion checks:**
- `echo one & echo two` yields two commands.
- `echo one&echo two` yields the same two commands.
- `echo one>first&echo two>second` associates each filename with the correct command.
- Spaces and tabs around separators do not become arguments.
- Empty or whitespace-only segments are handled safely; never dereference a missing command name.

The assignment does not specify every empty-segment edge case. Choose and document a consistent policy rather than inventing extra shell features. Do not implement pipes or interpret `&` as permission to skip waiting for the whole line.

## Problem 11 — Launch commands in parallel, then wait

Change execution so all external commands on one input line are launched before the shell waits for any of them.

**Your task:** Record the children you successfully create, then wait for all of them before printing the next prompt or reading the next batch line. Continue handling built-ins in the shell process, not as external programs.

**Completion checks:**
- `sleep 1 & sleep 1` takes roughly one second rather than two.
- `sleep 2 & echo ready` can print `ready` before the sleep finishes, but the shell waits for both before advancing.
- `echo one>first&echo two>second` produces the two separate files.
- An invalid command does not prevent already-started children from being reaped.
- A following batch line runs only after all children from the previous line finish.

Output order among parallel commands is not guaranteed. Do not add stronger ordering requirements than the assignment specifies.

## Problem 12 — Audit errors and resource lifetimes

Review the complete loop for error recovery, memory ownership, open files, and child processes. Fix gaps now rather than adding new features.

**Your task:**
- Check relevant operation results, including file opening, allocation, `fork()`, `execv()`, `chdir()`, redirection, and waiting.
- Distinguish EOF from a read failure.
- Use only the required message for shell-detected errors.
- Preserve startup failures' status 1 and normal EOF/valid `exit` status 0.
- Continue after ordinary command errors.
- Free per-line and replaced-path allocations when no longer needed.
- Close unneeded file descriptors and reap every successfully launched child.
- Ensure failed children cannot fall back into the shell loop.

**Completion checks:** Run many input lines, repeated `path` changes, long lines, invalid commands, and repeated redirections. Use compiler warnings and, if available, memory-checking tools to investigate leaks and invalid accesses. Keep previously working tests passing.

## Problem 13 — Run the final acceptance suite

Create repeatable tests covering the entire assignment. For each, record expected standard output, standard error, output-file contents, and exit status where relevant. Avoid depending on a particular ordering of parallel output.

**Required coverage:**
- Startup with zero, one, and too many arguments; unreadable/nonexistent and empty batch files.
- Exact interactive prompt `wish> `, no batch prompts, and graceful EOF in both modes.
- External commands with arguments; blank lines, tabs, repeated whitespace, long lines, and a final batch line without a newline.
- All built-ins, their argument rules, failed directory changes, and empty/replaced/ordered search paths.
- Valid and invalid redirection, truncation, both output streams, and unwritable destinations.
- Parallel execution, per-command redirection, and waiting before the next input line.
- Recovery after errors and no leftover debugging output.

**Project complete when:** All tests pass, the executable is named `wish`, execution uses `execv()` rather than `system()`, and the behavior matches the original specification. Keep the final source and tests under version control or save a clearly labeled working copy.

Pipes, job control, quoting, expansion, and direct absolute/relative executable paths are not required for this project.

In [8]:
! ls -lsah ../../ostep-homework

total 16K
4.0K drwxrwsr-x 33 solveit solveit 4.0K Feb  5  2026 .
4.0K drwxrwsr-x 45 solveit solveit 4.0K Nov 12 17:26 ..
   0 drwxrwsr-x  8 solveit solveit  211 Feb  5  2026 .git
8.0K -rw-rw-r--  1 solveit solveit 6.1K Feb  5  2026 README.md
   0 drwxrwsr-x  2 solveit solveit  131 Feb  5  2026 cpu-api
   0 drwxrwsr-x  2 solveit solveit   57 Feb  5  2026 cpu-intro
   0 drwxrwsr-x  2 solveit solveit   55 Feb  5  2026 cpu-sched
   0 drwxrwsr-x  2 solveit solveit   53 Feb  5  2026 cpu-sched-lottery
   0 drwxrwsr-x  2 solveit solveit   50 Feb  5  2026 cpu-sched-mlfq
   0 drwxrwsr-x  2 solveit solveit   51 Feb  5  2026 cpu-sched-multi
   0 drwxrwsr-x  2 solveit solveit   49 Feb  5  2026 dist-afs
   0 drwxrwsr-x  2 solveit solveit   58 Feb  5  2026 dist-nfs
   0 drwxrwsr-x  2 solveit solveit   36 Feb  5  2026 file-devices
   0 drwxrwsr-x  2 solveit solveit   77 Feb  5  2026 file-disks
   0 drwxrwsr-x  2 solveit solveit  210 Feb  5  2026 file-ffs
   0 drwxrwsr-x  2 solveit solveit   50 Feb  5 